# Côte d'Ivoire - Projet PROMIRE / ONU-REDD

3 · Émissions et absorptions forestières 2021-2025 - données d'activité et classeur de calcul
-------
Ce notebook prépare les données d'activité au niveau de détail requis pour le calcul des émissions et des absorptions, puis produit un classeur Excel **dynamique** (toutes les cellules de résultats sont des formules) :

| Composante | Données d'activité | Facteur |
|:--|:--|:--|
| Déforestation - biomasse et matière organique morte | Def : zone × forêt 2021 × utilisation 2025 | FE net = stock forestier − stock post-déforestation |
| Déforestation - sols minéraux | idem | GIEC éq. 2.25 : SOC_ref · (F − 1) sur D = 20 ans |
| Déforestation - combustion de la biomasse (CH₄, N₂O) | idem, × fraction brûlée | GIEC éq. 2.27 : MB · Cf · Gef · 10⁻³ × PRG |
| Dégradation | Deg : zone × forêt 2021 × forêt 2025 | stock forêt 2021 − stock forêt 2025 |
| Renforcement des stocks (gains) | Gain : zone × catégorie de gain | taux de croissance annuel |

**Chaîne de calcul**

1. **Activité** de chaque échantillon : sortie du notebook `1`.
2. **Catégories** de chaque échantillon : catégorie forestière en 2021 et en 2025, utilisation post-déforestation en 2025, catégorie de gain.
3. **Superficies** : estimateur stratifié par zone (post-stratification zone × strate, comme dans le notebook `2`) ; chaque transition est un domaine de la zone.
4. **Facteurs** : stocks de carbone, biomasse combustible, facteurs des sols, paramètres de combustion et taux de croissance (valeurs nationales du NERF 2024 et valeurs par défaut du GIEC), recalculés par formules.
5. **Émissions, absorptions et incertitudes** : propagation selon l'approche 1 du GIEC ; les sols et les gains accumulent leurs effets d'une année à l'autre (cohortes).

**Entrées** : `results/CIV_echantillons_activite_2021_2025.csv` (notebook `1`) et les zones phytogéographiques (`data/ZnePhytoGeo_CI.shp`).

**Sorties** (dossier `results/`), classeurs produits par le module `sbae_emissions_excel.py` :

- `3_emissions_and_removals.xlsx` : national et par zone ;
- `3_emissions_and_removals_PRE.xlsx` : territoire du PRE (Programme de Réductions des Émissions), section 12.

Les vérifications de la section 10 comparent les valeurs du classeur à des calculs indépendants.

In [1]:
import os
from statistics import NormalDist
import numpy as np
import pandas as pd
import geopandas as gpd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Paramètres

In [2]:
data_path  = 'results/CIV_echantillons_activite_2021_2025.csv'   # sortie du notebook 1
zones_path = 'data/ZnePhytoGeo_CI.shp'
xlsx_path  = 'results/3_emissions_and_removals.xlsx'

initial_year, final_year = 2021, 2025      # durée de la période T = 4 ans
years = list(range(initial_year, final_year + 1))
confidence    = 0.90     # niveau de confiance des résultats
ef_confidence = 0.90     # niveau de confiance des incertitudes saisies (NERF / IFN : IC 90 %)
forest_class  = 'Terres_Forestieres'

# Sols minéraux : sols LAC, climat tropical humide
socref   = 47.0          # tC/ha (GIEC tableau 2.3)
u_socref = 0.0           # incertitude non disponible : à compléter
soc_d    = 20            # années

# Les échantillons hors des zones phytogéographiques sont attribués à la zone la plus proche (comme dans le notebook 2)
chercher_la_zone_la_plus_proche = True

## 2. Échantillons et activité (sortie du notebook 1)

In [3]:
ad = pd.read_csv(data_path)
print(len(ad), 'échantillons')
ad.activity.value_counts()

12000 échantillons


activity
SNF      9705
SF       1324
SDegF     860
Def        84
Deg        17
Gain       10
Name: count, dtype: int64

In [4]:
# Zone phytogéographique de chaque échantillon (même règle que le notebook 2)
zones = gpd.read_file(zones_path)
gdf = gpd.GeoDataFrame(ad, geometry=gpd.points_from_xy(ad.center_lon, ad.center_lat), crs='EPSG:4326').to_crs('EPSG:32630')
gdf = gpd.sjoin_nearest(gdf, zones[['Zone', 'geometry']].to_crs('EPSG:32630'), how='left', distance_col='zdistance')
assert gdf.plotid.is_unique
ad = pd.DataFrame(gdf.drop(columns=['geometry', 'index_right'])).rename(columns={'Zone': 'zone'})
print("Échantillons hors des zones :", (ad.zdistance > 0).sum())
if not chercher_la_zone_la_plus_proche:
    ad = ad[ad.zdistance == 0]

# Échantillons valides pour l'estimateur : strate connue et activité déterminée (comme dans le notebook 2)
valid = ad[ad.strata.notna() & (ad.activity != 'ND')].copy()
valid['strata'] = valid['strata'].astype(int)
print(len(valid), 'échantillons valides')
valid.activity.value_counts()

Échantillons hors des zones : 178
11910 échantillons valides


activity
SNF      9633
SF       1310
SDegF     856
Def        84
Deg        17
Gain       10
Name: count, dtype: int64

## 3. Classes CEO de 2021 et de 2025

Pour chaque échantillon : classe de niveau 2 en 2021 (`class_initial`) et en 2025 (`class_final`). Selon l'activité, elles déterminent :

- **Def** : le stock forestier perdu (classe 2021) et l'utilisation qui suit (classe 2025, toujours non forestière) ;
- **Deg** : la forêt de 2021 et la forêt de 2025 ;
- **Gain** : le type de forêt qui s'installe (classe 2025).

In [5]:
valid['class_initial'] = valid[f'n2_{initial_year}']
valid['class_final'] = valid[f'n2_{final_year}']
is_def = valid.activity == 'Def'
assert (valid.loc[is_def, f'n1_{final_year}'] != forest_class).all()

# Année de l'événement de déforestation (première année non forestière) - information
valid['annee_def'] = valid[is_def].apply(
    lambda r: next((y for y in years[1:] if r[f'n1_{y}'] != forest_class), np.nan), axis=1).reindex(valid.index)

for activite in ['Def', 'Deg', 'Gain']:
    d = valid[valid.activity == activite]
    print(f'\n{activite} : {len(d)} échantillons (classe 2021 en lignes, classe 2025 en colonnes)')
    print(pd.crosstab(d.class_initial, d.class_final).to_string())


Def : 84 échantillons (classe 2021 en lignes, classe 2025 en colonnes)
class_final                Amenagement_Agricole  Anacarde  Cacao  Cafe  Infrastructures
class_initial                                                                          
Foret_Dense                                  22         1      1     0                1
Foret_Galerie                                 4         0      0     0                0
Foret_Secondaire_Degradee                    35         0      2     0                1
Foret_Sol_Hydromorphe                         1         0      0     0                0
Plantation_Agroforestiere                     5         0     10     1                0

Deg : 17 échantillons (classe 2021 en lignes, classe 2025 en colonnes)
class_final                Foret_Secondaire_Degradee  Plantation_Agroforestiere
class_initial                                                                  
Foret_Dense                                       11                          0


In [6]:
print("Répartition des échantillons Def par année d'événement (information) :")
valid.loc[is_def, 'annee_def'].astype(int).value_counts().sort_index()

Répartition des échantillons Def par année d'événement (information) :


annee_def
2022    18
2023    28
2024    23
2025    15
Name: count, dtype: int64

## 4. Correspondance des classes CEO avec les catégories

Les catégories sont celles des facteurs d'émission nationaux (NERF 2024). Ces tables sont reproduites dans la feuille « Correspondances » du classeur, où elles restent modifiables.

In [7]:
forest_cats = ['Forêt non dégradée', 'Forêt secondaire ou dégradée', 'Agroforêt']
post_cats = ['Cultures annuelles', 'Cacao', 'Hévéa', 'Palmier à huile', 'Cultures pérennes', 'Jachère / fourré',
             'Terres graminéennes', 'Sans biomasse']
gain_cats = ['Forêt secondaire', 'Agroforêt', 'Plantation / reboisement']

forest_map = {
    'Foret_Dense': 'Forêt non dégradée',
    'Foret_Galerie': 'Forêt non dégradée',
    'Foret_Sol_Hydromorphe': 'Forêt non dégradée',
    'Mangrove': 'Forêt non dégradée',
    'Foret_Secondaire_Degradee': 'Forêt secondaire ou dégradée',
    'Plantation_Agroforestiere': 'Agroforêt',
    'Plantation_Forestiere_Reboisement': 'Forêt secondaire ou dégradée',
}
post_map = {
    'Amenagement_Agricole': 'Cultures annuelles',
    'Cacao': 'Cacao',
    'Hevea': 'Hévéa',
    'Palmier': 'Palmier à huile',
    'Anacarde': 'Cultures pérennes',
    'Cafe': 'Cultures pérennes',
    'Cocoteraie': 'Cultures pérennes',
    'Plantation_Fruitiere': 'Cultures pérennes',
    'Jachere_Fourree': 'Jachère / fourré',
    'Savane_Arboree': 'Terres graminéennes',
    'Savane_Arbustive': 'Terres graminéennes',
    'Savane_Herbeuse': 'Terres graminéennes',
    'Habitation': 'Sans biomasse',
    'Infrastructures': 'Sans biomasse',
    'Affleurement_Rocheux': 'Sans biomasse',
    'Sol_Nu': 'Sans biomasse',
    'Cours_Voie_Eau': 'Sans biomasse',
    'Lac': 'Sans biomasse',
    'Plan_Eau': 'Sans biomasse',
    'Zones_Marecageuses': 'Sans biomasse',
}
gain_map = {
    'Foret_Dense': 'Forêt secondaire',
    'Foret_Galerie': 'Forêt secondaire',
    'Foret_Sol_Hydromorphe': 'Forêt secondaire',
    'Mangrove': 'Forêt secondaire',
    'Foret_Secondaire_Degradee': 'Forêt secondaire',
    'Plantation_Agroforestiere': 'Agroforêt',
    'Plantation_Forestiere_Reboisement': 'Plantation / reboisement',
}
mapping_notes = [
    'Forêt galerie, forêt sur sol hydromorphe et mangrove : stock de la forêt dense (forêt non dégradée) de la zone.',
    'Plantation forestière / reboisement : stock de la forêt secondaire par défaut. Pour les gains, taux de croissance propre (« Plantation / reboisement »).',
    'Jachère / fourré : stock de biomasse des terres graminéennes et facteurs des sols des cultures annuelles (catégorie « Fourré »).',
    'Anacarde, café, cocoteraie, plantation fruitière : stock des cultures pérennes (moyenne hévéa / palmier).',
    'Habitations, infrastructures, sol nu, affleurements, eaux et zones marécageuses : stock de biomasse nul et perte totale du carbone des sols (établissements humains et autres terres).',
    'Gains : toute forêt naturelle qui s\'installe croît au taux de la forêt secondaire de la zone.',
]

# Toutes les classes utilisées doivent avoir une catégorie
act = valid.activity
assert valid.loc[act.isin(['Def', 'Deg']), 'class_initial'].isin(forest_map).all()
assert valid.loc[act == 'Def', 'class_final'].isin(post_map).all()
assert valid.loc[act == 'Deg', 'class_final'].isin(forest_map).all()
assert valid.loc[act == 'Gain', 'class_final'].isin(gain_map).all()
assert set(forest_map.values()) <= set(forest_cats) and set(post_map.values()) <= set(post_cats)
assert set(gain_map.values()) <= set(gain_cats)
print('Toutes les classes des échantillons Def, Deg et Gain sont attribuées.')

Toutes les classes des échantillons Def, Deg et Gain sont attribuées.


## 5. Stocks de carbone

Valeurs nationales du NERF 2024. Les incertitudes sont des demi-largeurs d'IC à 90 % exprimées en fraction (14,58 % → 0,1458).

- **AGB, BGB** : inventaire forestier national (IFN), Tier 2, en tonnes de matière sèche par hectare, et leurs incertitudes.
- **Litière** : GIEC Tier 1 en tC/ha ; incertitude tirée de la borne supérieure de l'intervalle GIEC. Pour la forêt secondaire, litière proportionnelle à l'AGB par rapport à la forêt dense.
- **Bois mort** : IFN, quatre composantes et leurs erreurs relatives.
- **CF** = 0,47 ; incertitude = |0,44 − 0,47| / 0,47 = 6,4 % (GIEC 2006, tableau 4.3).

In [8]:
ZO, ZM, ZS, ZSS = 'Secteur Ombrophile', 'Secteur mésophile', 'Secteur Soudanais', 'Secteur Sub Soudanais'
FND, FSD, AGF = forest_cats


def fstock(cat, zone, agb, u_agb, bgb, u_bgb, dw, e, litter=None, src=''):
    """Une ligne de stock forestier. dw / e : (couché, souches, sur pied, racines sur pied) en tms/ha / fraction."""
    row = dict(category=cat, zone=zone, agb=agb, u_agb=u_agb, bgb=bgb, u_bgb=u_bgb,
               dw_lying=dw[0], dw_stump=dw[1], dw_standing=dw[2], dw_bgb_standing=dw[3],
               e_lying=e[0], e_stump=e[1], e_standing=e[2], e_bgb_standing=e[3], source=src,
               litter_rule='none', litter_tc=np.nan, litter_upper_tc=np.nan, litter_ref=None)
    if litter == 'scaled':
        row.update(litter_rule='scaled', litter_ref=FND)
    elif litter is not None:
        row.update(litter_rule='ipcc', litter_tc=litter[0], litter_upper_tc=litter[1])
    return row


NO_DW = ((0, 0, 0, 0), (0, 0, 0, 0))
forest_stocks = pd.DataFrame([
    # forêt non dégradée (forêt dense)
    fstock(FND, ZO,  204.57, .1458, 75.69, .1458, (16.96, .33, 13.59, 3.80), (.2624, .3162, .2627, .2360), (4.8, 16.4), 'NERF 2024 (IFN)'),
    fstock(FND, ZM,  134.70, .1688, 30.60, .1992, (13.95, .24, 14.36, 3.96), (.2903, .5316, .4321, .3852), (4.8, 16.4), 'NERF 2024 (IFN)'),
    fstock(FND, ZS,  104.30, .0939, 29.21, .0939, (3.01, .03, 5.67, 1.67), (.5082, 1.0007, .2662, .2494), (5.9, 14.8), 'NERF 2024 (IFN)'),
    fstock(FND, ZSS, 101.40, .3614, 22.43, .4334, (8.93, .64, 12.19, 3.57), (.5515, 1.0393, .4252, .3974), (5.9, 14.8), 'NERF 2024 (IFN)'),
    # forêt secondaire ou dégradée : litière proportionnelle à l'AGB
    fstock(FSD, ZO,  107.71, .1441, 39.85, .1441, (17.58, .50, 12.78, 3.64), (.2312, .2666, .2314, .2110), 'scaled', 'NERF 2024 (IFN)'),
    fstock(FSD, ZM,   67.89, .1417, 13.58, .1417, (27.55, .60, 21.11, 5.66), (.3933, .4660, .4026, .3611), 'scaled', 'NERF 2024 (IFN)'),
    fstock(FSD, ZS,   70.71, .2198, 19.80, .2198, (3.99, .10, 11.23, 3.11), (.4417, .9661, .4829, .4488), 'scaled', 'NERF 2024 (IFN)'),
    fstock(FSD, ZSS,  48.11, .2643,  9.62, .2643, (4.88, .14, 6.59, 1.94), (.4172, .9215, .7130, .6569), 'scaled', 'NERF 2024 (IFN)'),
    # agroforêt : identique pour toutes les zones, sans litière ni bois mort
    *[fstock(AGF, z, 45.80, .0954, 8.40, .1321, *NO_DW, None, 'NERF 2024 (IFN)') for z in (ZO, ZM, ZS, ZSS)],
])

post_stocks = pd.DataFrame([
    dict(category='Cultures annuelles',  rule='input', agb=5.53,   u_agb=.7505,  bgb=0.0,  u_bgb=0.0,   source='NERF 2024 (aménagement agricole)'),
    dict(category='Cacao',               rule='input', agb=37.2,   u_agb=.1282,  bgb=8.2,  u_bgb=.1207, source='NERF 2024'),
    dict(category='Hévéa',               rule='input', agb=113.4,  u_agb=.1828,  bgb=22.8, u_bgb=.1803, source='NERF 2024 (hévéa 10 ans)'),
    dict(category='Palmier à huile',     rule='input', agb=60.0,   u_agb=.2330,  bgb=12.0, u_bgb=.2333, source='NERF 2024 (palmier 20 ans)'),
    dict(category='Cultures pérennes',   rule='mean', mean_of=['Hévéa', 'Palmier à huile'], agb=np.nan, u_agb=np.nan,
         bgb=np.nan, u_bgb=np.nan, source='NERF 2024 (moyenne hévéa / palmier)'),
    dict(category='Jachère / fourré',    rule='mean', mean_of=['Terres graminéennes'], agb=np.nan, u_agb=np.nan,
         bgb=np.nan, u_bgb=np.nan, source='Stock des terres graminéennes (« Fourré »)'),
    dict(category='Terres graminéennes', rule='input', agb=35.33,  u_agb=1.2479, bgb=4.55, u_bgb=1.0593, source='NERF 2024'),
    dict(category='Sans biomasse',       rule='zero',  agb=0.0,    u_agb=0.0,    bgb=0.0,  u_bgb=0.0,   source='Stock nul (établissements humains, autres terres)'),
])
forest_stocks[['category', 'zone', 'agb', 'u_agb', 'bgb', 'u_bgb', 'litter_rule', 'litter_tc', 'litter_upper_tc']]

,category,zone,agb,u_agb,bgb,u_bgb,litter_rule,litter_tc,litter_upper_tc
0,Forêt non dégradée,Secteur Ombrophile,204.57,0.1458,75.69,0.1458,ipcc,4.8,16.4
1,Forêt non dégradée,Secteur mésophile,134.70,0.1688,30.60,0.1992,ipcc,4.8,16.4
2,Forêt non dégradée,Secteur Soudanais,104.30,0.0939,29.21,0.0939,ipcc,5.9,14.8
3,Forêt non dégradée,Secteur Sub Soudanais,101.40,0.3614,22.43,0.4334,ipcc,5.9,14.8
4,Forêt secondaire ou dégradée,Secteur Ombrophile,107.71,0.1441,39.85,0.1441,scaled,NaN,NaN
5,Forêt secondaire ou dégradée,Secteur mésophile,67.89,0.1417,13.58,0.1417,scaled,NaN,NaN
6,Forêt secondaire ou dégradée,Secteur Soudanais,70.71,0.2198,19.80,0.2198,scaled,NaN,NaN
7,Forêt secondaire ou dégradée,Secteur Sub Soudanais,48.11,0.2643,9.62,0.2643,scaled,NaN,NaN
8,Agroforêt,Secteur Ombrophile,45.80,0.0954,8.40,0.1321,none,NaN,NaN
9,Agroforêt,Secteur mésophile,45.80,0.0954,8.40,0.1321,none,NaN,NaN


## 6. Facteurs des sols minéraux

Équation 2.25 du GIEC : la variation du stock des sols vaut SOC_ref · (F − 1) sur D = 20 ans, avec F = F_LU · F_MG · F_I pour l'utilisation qui suit la déforestation (F = 1 pour la forêt). Facteurs du NERF 2024. Aucune incertitude n'est disponible pour ces facteurs : elles sont mises à 0 et peuvent être complétées dans le classeur.

In [9]:
def soc(cat, flu, fmg, fi, src):
    return dict(category=cat, flu=flu, u_flu=0.0, fmg=fmg, u_fmg=0.0, fi=fi, u_fi=0.0, source=src)


soc_factors = pd.DataFrame([
    soc('Cultures annuelles',  0.83, 1.04, 0.92, 'NERF 2024 : aménagement agricole'),
    soc('Cacao',               1.01, 1.04, 0.92, 'NERF 2024 : cacao'),
    soc('Hévéa',               1.01, 1.04, 0.92, 'NERF 2024 : hévéa'),
    soc('Palmier à huile',     1.01, 1.04, 0.92, 'Facteurs des cultures pérennes'),
    soc('Cultures pérennes',   1.01, 1.04, 0.92, 'NERF 2024 : agriculture pérenne, anacarde'),
    soc('Jachère / fourré',    0.83, 1.04, 0.92, 'NERF 2024 : fourré'),
    soc('Terres graminéennes', 1.00, 1.00, 1.00, 'NERF 2024 : terres graminéennes (pas de variation)'),
    soc('Sans biomasse',       0.00, 0.00, 0.00, 'NERF 2024 : établissements humains, autres terres (perte totale)'),
])
assert list(soc_factors.category) == post_cats
soc_factors

,category,flu,u_flu,fmg,u_fmg,fi,u_fi,source
0,Cultures annuelles,0.83,0.0,1.04,0.0,0.92,0.0,NERF 2024 : aménagement agricole
1,Cacao,1.01,0.0,1.04,0.0,0.92,0.0,NERF 2024 : cacao
2,Hévéa,1.01,0.0,1.04,0.0,0.92,0.0,NERF 2024 : hévéa
3,Palmier à huile,1.01,0.0,1.04,0.0,0.92,0.0,Facteurs des cultures pérennes
4,Cultures pérennes,1.01,0.0,1.04,0.0,0.92,0.0,"NERF 2024 : agriculture pérenne, anacarde"
5,Jachère / fourré,0.83,0.0,1.04,0.0,0.92,0.0,NERF 2024 : fourré
6,Terres graminéennes,1.00,0.0,1.00,0.0,1.00,0.0,NERF 2024 : terres graminéennes (pas de variat...
7,Sans biomasse,0.00,0.0,0.00,0.0,0.00,0.0,"NERF 2024 : établissements humains, autres ter..."


## 6 bis. Combustion de la biomasse lors de la déforestation

Équation 2.27 du GIEC (2006, vol. 4) : L = A_brûlée · MB · Cf · Gef · 10⁻³ (t de CH₄ ou de N₂O), puis × PRG (AR5).

- **MB**, biomasse combustible en tms/ha = AGB + litière + bois mort (sans les racines), calculée dans le classeur à partir des stocks de la section 5 ; la litière (tC/ha) est convertie en matière sèche avec la fraction de carbone de la litière, 0,37.
- **Cf** = 0,32 ; **Gef** = 6,8 g/kg (CH₄) et 0,2 g/kg (N₂O) ; **PRG** = 28 (CH₄) et 265 (N₂O).
- **Fraction brûlée** de la superficie déforestée : 0,5, sauf 0 pour les conversions en jachère / fourré ou en terres graminéennes et pour l'agroforêt convertie en cacao ou en cultures pérennes.
- Aucune incertitude n'est disponible pour ces paramètres : elles sont mises à 0 et peuvent être complétées dans le classeur. Les incertitudes des superficies et de MB sont propagées.

In [10]:
# Paramètres de la combustion (GIEC 2006 ; incertitudes non disponibles : à compléter)
fire = dict(cf_litter=0.37,                     # fraction de carbone de la litière
            comb=0.32, u_comb=0.0,              # facteur de combustion Cf
            gef_ch4=6.8, u_gef_ch4=0.0,         # g CH4 / kg ms brûlée
            gef_n2o=0.2, u_gef_n2o=0.0,         # g N2O / kg ms brûlée
            gwp_ch4=28, gwp_n2o=265,            # PRG AR5
            u_fire=0.0)                         # incertitude de la fraction brûlée

# Fraction brûlée par transition : 0,5 sauf conversions sans feu
sans_feu = {'Jachère / fourré', 'Terres graminéennes'}
fire_fraction = {(f, u): 0.0 if u in sans_feu or (f == AGF and u in ('Cacao', 'Cultures pérennes')) else 0.5
                 for f in forest_cats for u in post_cats}
pd.Series(fire_fraction).unstack()

,Cacao,Cultures annuelles,Cultures pérennes,Hévéa,Jachère / fourré,Palmier à huile,Sans biomasse,Terres graminéennes
Agroforêt,0.0,0.5,0.0,0.5,0.0,0.5,0.5,0.0
Forêt non dégradée,0.5,0.5,0.5,0.5,0.0,0.5,0.5,0.0
Forêt secondaire ou dégradée,0.5,0.5,0.5,0.5,0.0,0.5,0.5,0.0


## 7. Taux de croissance des gains

Croissance nette de la biomasse aérienne, en tC/ha/an, et incertitude (IC 90 %), valeurs du NERF 2024. Seule la croissance aérienne est comptée (R = 0) ; le classeur permet de saisir un rapport racines/tiges R. Pour les combinaisons zone × type de forêt sans valeur propre, la valeur la plus proche est reprise (même type de forêt, même zone écologique FAO lorsque c'est possible) ; elles sont signalées dans la colonne source.

In [11]:
def taux(cat, zone, rate, u_rate, src):
    return dict(category=cat, zone=zone, rate=rate, u_rate=u_rate, r=0.0, source=src)


u_soud = 1.645 * 1 / 2.9          # 1,645 · ET / moyenne (GIEC 2019 tableau 4.9 : 2,9 ± 1 tms/ha/an)
removal_rates = pd.DataFrame([
    taux('Forêt secondaire', ZO,  3.29,  .34,    'NERF 2024'),
    taux('Forêt secondaire', ZM,  2.35,  .34,    'NERF 2024'),
    taux('Forêt secondaire', ZS,  1.363, u_soud, 'NERF 2024 (2,9 tms × 0,47)'),
    taux('Forêt secondaire', ZSS, 1.363, u_soud, 'Valeur de la zone soudanaise (même zone écologique FAO)'),
    taux('Agroforêt', ZO,  3.16, .75, 'NERF 2024'),
    taux('Agroforêt', ZM,  3.16, .75, 'Valeur des autres zones'),
    taux('Agroforêt', ZS,  3.16, .75, 'Valeur des autres zones'),
    taux('Agroforêt', ZSS, 3.16, .75, 'NERF 2024'),
    taux('Plantation / reboisement', ZO,  4.23, .67, 'Valeur de la zone mésophile'),
    taux('Plantation / reboisement', ZM,  4.23, .67, 'NERF 2024'),
    taux('Plantation / reboisement', ZS,  4.23, .67, 'Valeur de la zone mésophile'),
    taux('Plantation / reboisement', ZSS, 4.23, .67, 'Valeur de la zone mésophile'),
])
removal_rates

,category,zone,rate,u_rate,r,source
0,Forêt secondaire,Secteur Ombrophile,3.290,0.340000,0.0,NERF 2024
1,Forêt secondaire,Secteur mésophile,2.350,0.340000,0.0,NERF 2024
2,Forêt secondaire,Secteur Soudanais,1.363,0.567241,0.0,"NERF 2024 (2,9 tms × 0,47)"
3,Forêt secondaire,Secteur Sub Soudanais,1.363,0.567241,0.0,Valeur de la zone soudanaise (même zone écolog...
4,Agroforêt,Secteur Ombrophile,3.160,0.750000,0.0,NERF 2024
5,Agroforêt,Secteur mésophile,3.160,0.750000,0.0,Valeur des autres zones
6,Agroforêt,Secteur Soudanais,3.160,0.750000,0.0,Valeur des autres zones
7,Agroforêt,Secteur Sub Soudanais,3.160,0.750000,0.0,NERF 2024
8,Plantation / reboisement,Secteur Ombrophile,4.230,0.670000,0.0,Valeur de la zone mésophile
9,Plantation / reboisement,Secteur mésophile,4.230,0.670000,0.0,NERF 2024


## 8. Superficies cartographiées des strates par zone

Mêmes valeurs que dans le notebook `2` (script GEE : https://code.earthengine.google.com/6d246c730ae5efc836ced21ef90d74ac).

In [12]:
area_hz = pd.DataFrame({
    'Secteur Ombrophile':    [28127.4082,	21377.56608,	8586.479557],
        'Secteur Soudanais':     [86493.21318,	39087.48208,	2729.667459],
        'Secteur Sub Soudanais': [21759.34011,	13883.2849,	1990.485385],
        'Secteur mésophile':     [54425.51251,	34889.54318,	7069.012614],
}, index=pd.Index([1, 2, 3], name='stratum')).T * 100      # km² -> ha
area_hz.index.name = 'zone'
area_hz = area_hz.loc[sorted(area_hz.index)]
area_hz

stratum,1,2,3
zone,,,
Secteur Ombrophile,2812740.820,2137756.608,858647.9557
Secteur Soudanais,8649321.318,3908748.208,272966.7459
Secteur Sub Soudanais,2175934.011,1388328.490,199048.5385
Secteur mésophile,5442551.251,3488954.318,706901.2614


## 9. Production du classeur

In [ ]:
import os,sys
# Check for xlsxwrite module, install it in /tmp if not installed
try:
    import xlsxwriter
except:
    %pip install xlsxwriter --target /tmp
    import sys, os
    sys.path.append(os.path.abspath('/tmp'))
    
from sbae_emissions_excel import build_emissions_workbook

samples = valid[['plotid', 'strata', 'zone', 'activity', 'class_initial', 'class_final', 'annee_def',
                 'center_lon', 'center_lat']].reset_index(drop=True)
os.makedirs(os.path.dirname(xlsx_path), exist_ok=True)
xl = build_emissions_workbook(
    xlsx_path, samples, area_hz,
    cats={'forest': forest_cats, 'post': post_cats, 'gain': gain_cats},
    maps={'forest': forest_map, 'post': post_map, 'gain': gain_map},
    forest_stocks=forest_stocks, post_stocks=post_stocks, soc_factors=soc_factors, removal_rates=removal_rates,
    confidence=confidence, ef_confidence=ef_confidence, initial_year=initial_year, final_year=final_year,
    socref=socref, u_socref=u_socref, soc_d=soc_d, fire=fire, fire_fraction=fire_fraction,
    sources={'data': os.path.basename(data_path),
             'areas': 'script GEE https://code.earthengine.google.com/6d246c730ae5efc836ced21ef90d74ac',
             'mapping_notes': mapping_notes},
    extra_cols=[("Année de l'événement", 'annee_def'), ('Longitude', 'center_lon'), ('Latitude', 'center_lat')])
print('Classeur écrit :', os.path.abspath(xlsx_path))

Classeur écrit : c:\Users\DOBLAS\OneDrive - Food and Agriculture Organization\Documents\code\sbae2xls_civ_2022_2025\results\3_emissions_and_removals.xlsx


## 10. Vérifications indépendantes

1. Les superficies du classeur sont identiques à celles de l'estimateur stratifié du notebook `2`, appliqué transition par transition.
2. Les émissions et absorptions de chaque composante sont recalculées ici directement à partir des superficies et des facteurs.

In [14]:
# 1. Superficies : estimateur stratifié indépendant (même fonction que dans le notebook 2)
def stratified_area_estimate(df, indicator, weights, total_area, z):
    n_h = df.groupby('strata').size()
    n_hk = df[indicator].groupby(df['strata']).sum().reindex(n_h.index, fill_value=0)
    p_h = n_hk / n_h
    w = weights.reindex(n_h.index)
    p = (w * p_h).sum()
    se = np.sqrt((w ** 2 * p_h * (1 - p_h) / (n_h - 1)).sum()) * total_area
    return p * total_area, se


z = round(NormalDist().inv_cdf(1 - (1 - confidence) / 2), 3)
T = final_year - initial_year


def verifier_superficies(rows, activite, condition):
    for _, r in rows.iterrows():
        dz = samples[samples.zone == r.zone].copy()
        dz['k'] = ((dz.activity == activite) & condition(dz, r)).astype(int)
        a_h = area_hz.loc[r.zone]
        area, se = stratified_area_estimate(dz, 'k', a_h / a_h.sum(), a_h.sum(), z)
        assert np.isclose(area, r.area) and np.isclose(se, r.se), (activite, r.key)


verifier_superficies(xl.def_rows, 'Def', lambda d, r: (d.class_initial.map(forest_map) == r.forest)
                     & (d.class_final.map(post_map) == r.post))
verifier_superficies(xl.deg_rows, 'Deg', lambda d, r: (d.class_initial.map(forest_map) == r.f_i)
                     & (d.class_final.map(forest_map) == r.f_f))
verifier_superficies(xl.gain_rows, 'Gain', lambda d, r: d.class_final.map(gain_map) == r.gain)
for activite in ['Def', 'Deg', 'Gain']:
    zd = xl.zone_act[activite].set_index('zone')
    for zn in area_hz.index:
        dz = samples[samples.zone == zn].assign(k=lambda d: (d.activity == activite).astype(int))
        a_h = area_hz.loc[zn]
        area, se = stratified_area_estimate(dz, 'k', a_h / a_h.sum(), a_h.sum(), z)
        assert np.isclose(area, zd.loc[zn, 'area']) and np.isclose(se, zd.loc[zn, 'se'])
print('Superficies par transition et par zone (Def, Deg, Gain) : identiques à l\'estimateur indépendant.')

Superficies par transition et par zone (Def, Deg, Gain) : identiques à l'estimateur indépendant.


In [15]:
# 2. Émissions et absorptions recalculées directement
fs = {k: v['ef'] for k, v in xl.forest_stocks.items()}
ps = {k: v['ef'] for k, v in xl.post_stocks.items()}
F = {r.category: r.flu * r.fmg * r.fi for r in soc_factors.itertuples()}
G = {(r.category, r.zone): -r.rate * (1 + r.r) * 44 / 12 for r in removal_rates.itertuples()}
cohortes = (T + 1) / 2

d = xl.def_rows
assert np.allclose(d.em_net, d.area * (d.apply(lambda r: fs[(r.forest, r.zone)], axis=1) - d.post.map(ps)))
s = xl.soc_rows
assert np.allclose(s.em, s.area * (-socref * (s.post.map(F) - 1) / soc_d * 44 / 12) * cohortes)
# combustion : A · fraction brûlée · MB · Cf · Gef · 10⁻³ · PRG, avec MB = AGB + litière / 0,37 + bois mort
mb = {k: r.agb + v['litter'] / fire['cf_litter'] + v['dw']
      for k, v in xl.forest_stocks.items() for r in [forest_stocks.set_index(['category', 'zone']).loc[k]]}
brule = d.area * d.apply(lambda r: fire_fraction[(r.forest, r.post)] * mb[(r.forest, r.zone)], axis=1) * fire['comb']
for gaz in ['ch4', 'n2o']:
    assert np.allclose(xl.fire_rows[f'em_{gaz}'], brule * fire[f'gef_{gaz}'] * 1e-3 * fire[f'gwp_{gaz}'])
g = xl.deg_rows
assert np.allclose(g.em, g.area * g.apply(lambda r: fs[(r.f_i, r.zone)] - fs[(r.f_f, r.zone)], axis=1))
a = xl.gain_rows
assert np.allclose(a.em, a.area * a.apply(lambda r: G[(r.gain, r.zone)], axis=1) * cohortes)
assert np.allclose(xl.direct.set_index('zone').em_net, d.groupby('zone').em_net.sum())
assert np.isclose(xl.years[['def', 'soc', 'ch4', 'n2o', 'deg', 'rscf']].sum().sum(), xl.bilan['net']['period'])
print('Émissions DEF (biomasse, sols, combustion CH₄ et N₂O), dégradation et absorptions : identiques au calcul direct ; '
      'série annuelle cohérente avec le total de la période.')

Émissions DEF (biomasse, sols, combustion CH₄ et N₂O), dégradation et absorptions : identiques au calcul direct ; série annuelle cohérente avec le total de la période.


## 11. Résultats

Moyennes annuelles sur la période ; incertitudes à 90 % exprimées en % de la valeur et en valeur absolue (±). Les absorptions sont négatives.

In [16]:
noms = {'def': 'Déforestation - biomasse et MOM (FE net)', 'soc': 'Déforestation - sols minéraux',
        'ch4': 'Déforestation - combustion de la biomasse, CH4', 'n2o': 'Déforestation - combustion de la biomasse, N2O',
        'deg': 'Dégradation', 'tot': 'Total des émissions', 'rscf': 'Renforcement des stocks (RSCF)',
        'net': 'Bilan net'}
bilan = pd.DataFrame([{'composante': noms[c], 'tCO2e/an': v['year'], 'incertitude': v['u'], '± tCO2e/an': v['pm'],
                       'total période (tCO2e)': v['period']} for c, v in xl.bilan.items() if c in noms]).set_index('composante')
bilan = bilan.loc[[noms[c] for c in ['def', 'soc', 'ch4', 'n2o', 'deg', 'tot', 'rscf', 'net']]]
bilan.style.format({'tCO2e/an': '{:,.0f}', 'incertitude': '{:.1%}', '± tCO2e/an': '{:,.0f}',
                    'total période (tCO2e)': '{:,.0f}'})

,tCO2e/an,incertitude,± tCO2e/an,total période (tCO2e)
composante,,,,
Déforestation - biomasse et MOM (FE net),"10,603,010",21.4%,"2,264,318","42,412,038"
Déforestation - sols minéraux,"223,710",22.5%,"50,360","894,840"
"Déforestation - combustion de la biomasse, CH4","169,362",21.8%,"36,923","677,447"
"Déforestation - combustion de la biomasse, N2O","47,144",21.8%,"10,278","188,575"
Dégradation,"1,336,232",49.2%,"657,554","5,344,927"
Total des émissions,"12,379,457",19.1%,"2,358,711","49,517,827"
Renforcement des stocks (RSCF),"-183,834",88.9%,"163,498","-735,337"
Bilan net,"12,195,622",19.4%,"2,364,371","48,782,490"


In [17]:
print('Superficies annuelles (estimation directe) :')
sup = pd.DataFrame([{'activité': a, 'ha/an': n['area_year'], 'incertitude': n['u_area'], '± ha/an': n['u_area'] * n['area_year']}
                    for a, n in ((a, xl.zone_act[a].attrs['national']) for a in ['Def', 'Deg', 'Gain'])]).set_index('activité')
sup.style.format({'ha/an': '{:,.0f}', 'incertitude': '{:.1%}', '± ha/an': '{:,.0f}'})

Superficies annuelles (estimation directe) :


,ha/an,incertitude,± ha/an
activité,,,
Def,"53,536",18.2%,"9,728"
Deg,"10,915",40.6%,"4,434"
Gain,"6,455",52.7%,"3,404"


In [18]:
print('Bilan par zone (tCO2e/an) :')
composantes = ['def', 'soc', 'ch4', 'n2o', 'deg', 'rscf', 'net']
lignes = []
for zn, comp in list(xl.bilan_zone.items()) + [('National', xl.bilan)]:
    ligne = {'zone': zn}
    for c in composantes:
        ligne[c] = comp[c]['year']
        ligne[f'U {c}'] = comp[c]['u']
    lignes.append(ligne)
par_zone = pd.DataFrame(lignes).set_index('zone')
par_zone.style.format({c: '{:,.0f}' for c in composantes} | {f'U {c}': '{:.1%}' for c in composantes})

Bilan par zone (tCO2e/an) :


,def,U def,soc,U soc,ch4,U ch4,n2o,U n2o,deg,U deg,rscf,U rscf,net,U net
zone,,,,,,,,,,,,,,
Secteur Ombrophile,"1,349,247",59.1%,"16,013",61.2%,"20,545",60.1%,"5,719",60.1%,"481,771",87.8%,0,0.0%,"1,873,295",48.2%
Secteur Soudanais,"5,263,443",31.2%,"107,901",31.7%,"81,164",32.6%,"22,593",32.6%,"236,230",109.5%,0,0.0%,"5,711,331",29.1%
Secteur Sub Soudanais,"977,511",62.4%,"23,331",57.7%,"18,073",61.5%,"5,031",61.5%,"129,861",141.5%,0,0.0%,"1,153,806",55.3%
Secteur mésophile,"3,012,808",39.5%,"76,466",43.2%,"49,580",39.7%,"13,801",39.7%,"488,370",80.0%,"-183,834",88.9%,"3,457,191",36.6%
National,"10,603,010",21.4%,"223,710",22.5%,"169,362",21.8%,"47,144",21.8%,"1,336,232",49.2%,"-183,834",88.9%,"12,195,622",19.4%


In [19]:
print("Série par année (les sols et les absorptions s'accumulent d'une année à l'autre) :")
cols = ['def', 'soc', 'ch4', 'n2o', 'deg', 'tot', 'rscf', 'net', 'pm_net']
serie = xl.years.set_index('year')[cols]
serie['U net'] = serie.pm_net / serie.net.abs()
serie.style.format({c: '{:,.0f}' for c in cols} | {'U net': '{:.1%}'})

Série par année (les sols et les absorptions s'accumulent d'une année à l'autre) :


,def,soc,ch4,n2o,deg,tot,rscf,net,pm_net,U net
year,,,,,,,,,,
2022,"10,603,010","89,484","169,362","47,144","1,336,232","12,245,231","-73,534","12,171,697","2,359,166",19.4%
2023,"10,603,010","178,968","169,362","47,144","1,336,232","12,334,715","-147,067","12,187,647","2,362,142",19.4%
2024,"10,603,010","268,452","169,362","47,144","1,336,232","12,424,199","-220,601","12,203,598","2,367,093",19.4%
2025,"10,603,010","357,936","169,362","47,144","1,336,232","12,513,683","-294,135","12,219,548","2,374,006",19.4%


## 12. Estimations pour le PRE

Même chaîne de calcul, appliquée aux seuls échantillons situés dans le PRE (colonne `pre` produite par le notebook `1`), avec les superficies cartographiées des strates **dans le PRE**, par zone. Facteurs, correspondances et paramètres sont ceux des sections 4 à 7. Le total du PRE est la somme de ses zones.

Le PRE ne recoupe pas le Secteur Soudanais. Une zone du PRE sans aucun échantillon ne peut pas être estimée : elle est retirée (signalé ci-dessous).

In [20]:
xlsx_path_pre = 'results/3_emissions_and_removals_PRE.xlsx'

# Superficie cartographiée de chaque strate dans le PRE, par zone (km² -> ha)
area_hz_pre = pd.DataFrame({
    'Secteur Ombrophile':    [17262.20123016427,  12174.209957087007, 4988.553467034733],
    'Secteur Sub Soudanais': [24.385942521542056, 9.404869314886117,  0.24926422661324296],
    'Secteur mésophile':     [6623.2923265831605, 4158.777891905002,  710.699415400892],
}, index=pd.Index([1, 2, 3], name='stratum')).T * 100
area_hz_pre.index.name = 'zone'
area_hz_pre = area_hz_pre.loc[sorted(area_hz_pre.index)]

samples_pre = samples[samples.plotid.isin(valid.loc[valid['pre'], 'plotid'])].reset_index(drop=True)
assert ZS not in set(samples_pre.zone), 'échantillon du PRE attribué au Secteur Soudanais'
print(len(samples_pre), 'échantillons valides dans le PRE')

zones_sans_echantillon = [zn for zn in area_hz_pre.index if zn not in set(samples_pre.zone)]
for zn in zones_sans_echantillon:
    print(f"{zn} : aucun échantillon pour {area_hz_pre.loc[zn].sum():,.0f} ha "
          f"({area_hz_pre.loc[zn].sum() / area_hz_pre.to_numpy().sum():.2%} du PRE) -> zone retirée")
area_hz_pre = area_hz_pre.drop(index=zones_sans_echantillon)
pd.crosstab(samples_pre.activity, samples_pre.zone, margins=True)

1861 échantillons valides dans le PRE
Secteur Sub Soudanais : aucun échantillon pour 3,404 ha (0.07% du PRE) -> zone retirée


zone,Secteur Ombrophile,Secteur mésophile,All
activity,,,
Def,3,1,4
Deg,1,0,1
Gain,0,2,2
SDegF,139,73,212
SF,339,4,343
SNF,951,348,1299
All,1433,428,1861


In [21]:
xl_pre = build_emissions_workbook(
    xlsx_path_pre, samples_pre, area_hz_pre,
    cats={'forest': forest_cats, 'post': post_cats, 'gain': gain_cats},
    maps={'forest': forest_map, 'post': post_map, 'gain': gain_map},
    forest_stocks=forest_stocks, post_stocks=post_stocks, soc_factors=soc_factors, removal_rates=removal_rates,
    confidence=confidence, ef_confidence=ef_confidence, initial_year=initial_year, final_year=final_year,
    socref=socref, u_socref=u_socref, soc_d=soc_d, fire=fire, fire_fraction=fire_fraction,
    title="Côte d'Ivoire — PRE — Émissions et absorptions forestières", territory='PRE',
    sources={'data': os.path.basename(data_path) + ' (échantillons situés dans le PRE)',
             'areas': 'superficies des strates dans le PRE, par zone (fournies)'
                      + (f" ; zone sans échantillon retirée : {', '.join(zones_sans_echantillon)}" if zones_sans_echantillon else ''),
             'mapping_notes': mapping_notes},
    extra_cols=[("Année de l'événement", 'annee_def'), ('Longitude', 'center_lon'), ('Latitude', 'center_lat')])
print('Classeur écrit :', os.path.abspath(xlsx_path_pre))

Classeur écrit : c:\Users\DOBLAS\OneDrive - Food and Agriculture Organization\Documents\code\sbae2xls_civ_2022_2025\results\3_emissions_and_removals_PRE.xlsx


In [22]:
# Contrôle : superficies par zone et émissions du PRE vs calcul indépendant
for activite in ['Def', 'Deg', 'Gain']:
    zd = xl_pre.zone_act[activite].set_index('zone')
    for zn in area_hz_pre.index:
        dz = samples_pre[samples_pre.zone == zn].assign(k=lambda d: (d.activity == activite).astype(int))
        a_h = area_hz_pre.loc[zn]
        area, se = stratified_area_estimate(dz, 'k', a_h / a_h.sum(), a_h.sum(), z)
        assert np.isclose(area, zd.loc[zn, 'area']) and np.isclose(se, zd.loc[zn, 'se'])
d = xl_pre.def_rows
assert np.allclose(d.em_net, d.area * (d.apply(lambda r: fs[(r.forest, r.zone)], axis=1) - d.post.map(ps)))
g = xl_pre.deg_rows
assert np.allclose(g.em, g.area * g.apply(lambda r: fs[(r.f_i, r.zone)] - fs[(r.f_f, r.zone)], axis=1))
a = xl_pre.gain_rows
assert np.allclose(a.em, a.area * a.apply(lambda r: G[(r.gain, r.zone)], axis=1) * cohortes)
assert np.isclose(xl_pre.years[['def', 'soc', 'ch4', 'n2o', 'deg', 'rscf']].sum().sum(), xl_pre.bilan['net']['period'])
print('PRE : superficies (Def, Deg, Gain) et émissions DEF, DEG et RSCF identiques au calcul indépendant.')

PRE : superficies (Def, Deg, Gain) et émissions DEF, DEG et RSCF identiques au calcul indépendant.


In [23]:
print('PRE : moyennes annuelles sur la période (tCO2e/an) :')
bilan_pre = pd.DataFrame([{'composante': noms[c], 'tCO2e/an': v['year'], 'incertitude': v['u'], '± tCO2e/an': v['pm'],
                           'total période (tCO2e)': v['period']} for c, v in xl_pre.bilan.items() if c in noms]).set_index('composante')
bilan_pre = bilan_pre.loc[[noms[c] for c in ['def', 'soc', 'ch4', 'n2o', 'deg', 'tot', 'rscf', 'net']]]
bilan_pre.style.format({'tCO2e/an': '{:,.0f}', 'incertitude': '{:.1%}', '± tCO2e/an': '{:,.0f}',
                        'total période (tCO2e)': '{:,.0f}'})

PRE : moyennes annuelles sur la période (tCO2e/an) :


,tCO2e/an,incertitude,± tCO2e/an,total période (tCO2e)
composante,,,,
Déforestation - biomasse et MOM (FE net),"698,984",85.5%,"597,875","2,795,936"
Déforestation - sols minéraux,"10,660",84.7%,"9,033","42,638"
"Déforestation - combustion de la biomasse, CH4","10,500",85.6%,"8,984","42,001"
"Déforestation - combustion de la biomasse, N2O","2,923",85.6%,"2,501","11,691"
Dégradation,"81,538",168.3%,"137,206","326,152"
Total des émissions,"804,605",76.3%,"613,554","3,218,419"
Renforcement des stocks (RSCF),"-40,975",138.1%,"56,573","-163,899"
Bilan net,"763,630",80.7%,"616,156","3,054,520"


In [24]:
print('PRE : superficies annuelles (estimation directe) :')
sup_pre = pd.DataFrame([{'activité': a, 'ha/an': n['area_year'], 'incertitude': n['u_area'], '± ha/an': n['u_area'] * n['area_year']}
                        for a, n in ((a, xl_pre.zone_act[a].attrs['national']) for a in ['Def', 'Deg', 'Gain'])]).set_index('activité')
sup_pre.style.format({'ha/an': '{:,.0f}', 'incertitude': '{:.1%}', '± ha/an': '{:,.0f}'})

PRE : superficies annuelles (estimation directe) :


,ha/an,incertitude,± ha/an
activité,,,
Def,"2,404",84.7%,"2,037"
Deg,344,164.5%,565
Gain,"1,415",115.9%,"1,640"


In [ ]:
print('PRE : bilan par zone (tCO2e/an) :')
lignes = []
for zn, comp in list(xl_pre.bilan_zone.items()) + [('PRE', xl_pre.bilan)]:
    ligne = {'zone': zn}
    for c in composantes:
        ligne[c] = comp[c]['year']
        ligne[f'U {c}'] = comp[c]['u']
    lignes.append(ligne)
par_zone_pre = pd.DataFrame(lignes).set_index('zone')
par_zone_pre.style.format({c: '{:,.0f}' for c in composantes} | {f'U {c}': '{:.1%}' for c in composantes})